<a href="https://colab.research.google.com/github/SamarthD07/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SamarthD07/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Rule: Prioritize published pages that had enough GSC exposure and clicks in February to make the signal meaningful, then rank pages higher when they show a strong refresh signal from historical search performance. The score uses simple, transparent conditions with no fitted weights and uses only information available by the end of February.

Reason codes:

* strong_refresh_signal — page meets the strongest combination of the selected pre-February signals.
* moderate_refresh_signal — page meets part of the refresh criteria.
* weak_signal — page qualifies for review but has limited supporting evidence.
* no_clear_signal — available February signals do not provide a strong refresh indication.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [9]:
from google.colab import userdata
from huggingface_hub import hf_hub_download
import pandas as pd

HF_TOKEN = userdata.get("HF_TOKEN")

feb_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-02/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

feb = pd.read_parquet(feb_file)

print("February data loaded")
print("Rows:", len(feb))
print("Date range:", feb["report_date"].min(), "to", feb["report_date"].max())
print("Columns:", len(feb.columns))

February data loaded
Rows: 7355108
Date range: 2026-02-01 to 2026-02-28
Columns: 30


In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from huggingface_hub import hf_hub_download
import pandas as pd
import numpy as np
import os

# Load only the columns needed for the baseline rule.
needed_cols = [
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position"
]

feb_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-02/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

df = pd.read_parquet(feb_file, columns=needed_cols)

print("February rows:", len(df))
print("Date range:", df["report_date"].min(), "to", df["report_date"].max())

# Fill missing signal values only for scoring.
impressions = df["gsc_impressions"].fillna(0)
clicks = df["gsc_clicks"].fillna(0)

# Transparent rule-based score.
# Exposure = 2 points
# Click = 2 points
# Good search position (<=20) = 1 point
position_signal = (
    df["gsc_avg_position"].notna()
    & (df["gsc_avg_position"] <= 20)
)

df["action_score"] = (
    2 * (impressions > 0).astype("int8")
    + 2 * (clicks > 0).astype("int8")
    + 1 * position_signal.astype("int8")
).astype("int8")

# Reason code.
df["reason_code"] = np.select(
    [
        (impressions > 0) & (clicks > 0) & position_signal,
        (impressions > 0) & (clicks > 0),
        (impressions > 0) | (clicks > 0)
    ],
    [
        "strong_refresh_signal",
        "moderate_refresh_signal",
        "weak_signal"
    ],
    default="no_clear_signal"
)

# Rank using only the columns needed for ranking.
df = df.sort_values(
    ["action_score", "gsc_impressions", "gsc_clicks"],
    ascending=[False, False, False],
    kind="mergesort"
).reset_index(drop=True)

df["rank"] = np.arange(1, len(df) + 1, dtype="int64")

# Save required CSV.
os.makedirs("work/outputs", exist_ok=True)

output_cols = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "report_date",
    "action_score",
    "reason_code",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position"
]

output_path = "work/outputs/baseline_action_score.csv"
df[output_cols].to_csv(output_path, index=False)

print("\nRows ranked:", len(df))
print("Output:", output_path)

print("\nTop 10:")
display(df[output_cols].head(10))

February rows: 7355108
Date range: 2026-02-01 to 2026-02-28

Rows ranked: 7355108
Output: work/outputs/baseline_action_score.csv

Top 10:


,rank,client_hash_id,content_hash_id,report_date,action_score,reason_code,gsc_impressions,gsc_clicks,gsc_avg_position
0,1,client_23a62021009f63c4,content_44f34c0a90047651,2026-02-16,5,strong_refresh_signal,52631.0,2.0,0.018164
1,2,client_20259bd6705d81d4,content_e9d938ad87afe733,2026-02-20,5,strong_refresh_signal,27761.0,1.0,0.105111
2,3,client_23a62021009f63c4,content_f012d6908fc807da,2026-02-16,5,strong_refresh_signal,24797.0,15.0,19.844739
3,4,client_23a62021009f63c4,content_cf1af462f6317c31,2026-02-16,5,strong_refresh_signal,23614.0,12.0,18.541797
4,5,client_23a62021009f63c4,content_4fe94bdfd75c38f9,2026-02-17,5,strong_refresh_signal,21533.0,2.0,0.072633
5,6,client_73cda7b4e4f265ea,content_8e1334d6356668e3,2026-02-23,5,strong_refresh_signal,18043.0,2.0,0.035194
6,7,client_23a62021009f63c4,content_44f34c0a90047651,2026-02-19,5,strong_refresh_signal,16661.0,1.0,0.055219
7,8,client_73cda7b4e4f265ea,content_252aa5480bb1f8d7,2026-02-09,5,strong_refresh_signal,15563.0,1.0,2.890445
8,9,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,2026-02-25,5,strong_refresh_signal,14576.0,1.0,0.003842
9,10,client_23a62021009f63c4,content_4fe94bdfd75c38f9,2026-02-23,5,strong_refresh_signal,14376.0,1.0,0.198943


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
display(df[[
    "rank",
    "client_hash_id",
    "content_hash_id",
    "report_date",
    "action_score",
    "reason_code",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position"
]].head(20))

,rank,client_hash_id,content_hash_id,report_date,action_score,reason_code,gsc_impressions,gsc_clicks,gsc_avg_position
0,1,client_23a62021009f63c4,content_44f34c0a90047651,2026-02-16,5,strong_refresh_signal,52631.0,2.0,0.018164
1,2,client_20259bd6705d81d4,content_e9d938ad87afe733,2026-02-20,5,strong_refresh_signal,27761.0,1.0,0.105111
2,3,client_23a62021009f63c4,content_f012d6908fc807da,2026-02-16,5,strong_refresh_signal,24797.0,15.0,19.844739
3,4,client_23a62021009f63c4,content_cf1af462f6317c31,2026-02-16,5,strong_refresh_signal,23614.0,12.0,18.541797
4,5,client_23a62021009f63c4,content_4fe94bdfd75c38f9,2026-02-17,5,strong_refresh_signal,21533.0,2.0,0.072633
5,6,client_73cda7b4e4f265ea,content_8e1334d6356668e3,2026-02-23,5,strong_refresh_signal,18043.0,2.0,0.035194
6,7,client_23a62021009f63c4,content_44f34c0a90047651,2026-02-19,5,strong_refresh_signal,16661.0,1.0,0.055219
7,8,client_73cda7b4e4f265ea,content_252aa5480bb1f8d7,2026-02-09,5,strong_refresh_signal,15563.0,1.0,2.890445
8,9,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,2026-02-25,5,strong_refresh_signal,14576.0,1.0,0.003842
9,10,client_23a62021009f63c4,content_4fe94bdfd75c38f9,2026-02-23,5,strong_refresh_signal,14376.0,1.0,0.198943


In [12]:
# Top-20 review
# Review is based only on February/pre-label signals.

top20 = df.head(20).copy()

def confidence_note(row):
    if row["gsc_impressions"] >= 10000 and row["gsc_clicks"] >= 1:
        return "High: strong search exposure with observed clicks."
    elif row["gsc_impressions"] >= 1000:
        return "Medium-high: meaningful search exposure, but click evidence is limited."
    else:
        return "Medium: signal exists but exposure is comparatively limited."

def what_would_make_it_wrong(row):
    if row["gsc_clicks"] == 0:
        return "Could be wrong if the exposure is not representative or tracking is incomplete."
    elif row["gsc_avg_position"] > 20:
        return "Could be wrong if high exposure is driven by lower-value queries or position is unstable."
    else:
        return "Could be wrong if GSC tracking is incomplete or the February signal does not represent current page quality."

top20["action"] = "review_for_refresh"
top20["confidence_note"] = top20.apply(confidence_note, axis=1)
top20["what_would_make_it_wrong"] = top20.apply(
    what_would_make_it_wrong, axis=1
)

review_cols = [
    "rank",
    "action",
    "reason_code",
    "confidence_note",
    "what_would_make_it_wrong"
]

print("Top-20 review:")
display(top20[review_cols])

Top-20 review:


,rank,action,reason_code,confidence_note,what_would_make_it_wrong
0,1,review_for_refresh,strong_refresh_signal,High: strong search exposure with observed cli...,Could be wrong if GSC tracking is incomplete o...
1,2,review_for_refresh,strong_refresh_signal,High: strong search exposure with observed cli...,Could be wrong if GSC tracking is incomplete o...
2,3,review_for_refresh,strong_refresh_signal,High: strong search exposure with observed cli...,Could be wrong if GSC tracking is incomplete o...
3,4,review_for_refresh,strong_refresh_signal,High: strong search exposure with observed cli...,Could be wrong if GSC tracking is incomplete o...
4,5,review_for_refresh,strong_refresh_signal,High: strong search exposure with observed cli...,Could be wrong if GSC tracking is incomplete o...
5,6,review_for_refresh,strong_refresh_signal,High: strong search exposure with observed cli...,Could be wrong if GSC tracking is incomplete o...
6,7,review_for_refresh,strong_refresh_signal,High: strong search exposure with observed cli...,Could be wrong if GSC tracking is incomplete o...
7,8,review_for_refresh,strong_refresh_signal,High: strong search exposure with observed cli...,Could be wrong if GSC tracking is incomplete o...
8,9,review_for_refresh,strong_refresh_signal,High: strong search exposure with observed cli...,Could be wrong if GSC tracking is incomplete o...
9,10,review_for_refresh,strong_refresh_signal,High: strong search exposure with observed cli...,Could be wrong if GSC tracking is incomplete o...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Weak picks + leakage check

# 1. Inspect the weakest-scoring rows.
weak_picks = df.sort_values(
    ["action_score", "gsc_impressions", "gsc_clicks"],
    ascending=[True, True, True]
).head(10).copy()

print("Weakest picks:")
display(weak_picks[[
    "rank",
    "client_hash_id",
    "content_hash_id",
    "report_date",
    "action_score",
    "reason_code",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position"
]])

# 2. Explain why these can be questionable.
print("\nWhy these picks may be weak:")
print(
    "Low-scoring rows have limited or no supporting GSC exposure/click "
    "signals. They should therefore be treated as lower-confidence review "
    "candidates rather than guaranteed refresh opportunities."
)

# 3. Leakage check: scoring must use February only.
print("\nLeakage checks:")

min_date = df["report_date"].min()
max_date = df["report_date"].max()

print("Scoring date range:", min_date, "to", max_date)

feb_only = (
    min_date >= pd.Timestamp("2026-02-01").date()
    and max_date <= pd.Timestamp("2026-02-28").date()
)

print("February-only scoring data:", feb_only)

print(
    "Scoring fields:",
    [
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position"
    ]
)

print(
    "\nLeakage conclusion:",
    "No March/future-window fields were used in the baseline score."
)

print(
    "Product-flag check: no product flags were used in the scoring rule."
)

Weakest picks:


,rank,client_hash_id,content_hash_id,report_date,action_score,reason_code,gsc_impressions,gsc_clicks,gsc_avg_position
2621783,2621784,client_e547b89c05043229,content_4170f9c4f674077a,2026-02-01,0,no_clear_signal,0.0,0.0,NaN
2621784,2621785,client_e547b89c05043229,content_90ac8345dd6d7860,2026-02-01,0,no_clear_signal,0.0,0.0,NaN
2621785,2621786,client_e547b89c05043229,content_dbd97e21e1cfd688,2026-02-01,0,no_clear_signal,0.0,0.0,NaN
2621786,2621787,client_e547b89c05043229,content_415db4b1abba5360,2026-02-01,0,no_clear_signal,0.0,0.0,NaN
2621787,2621788,client_e547b89c05043229,content_b6c544aeb97a917d,2026-02-01,0,no_clear_signal,0.0,0.0,NaN
2621788,2621789,client_e547b89c05043229,content_d8f857aeef5cea81,2026-02-01,0,no_clear_signal,0.0,0.0,NaN
2621789,2621790,client_e547b89c05043229,content_5907dd335197d140,2026-02-01,0,no_clear_signal,0.0,0.0,NaN
2621790,2621791,client_e547b89c05043229,content_1b7ce71d7808c2fc,2026-02-01,0,no_clear_signal,0.0,0.0,NaN
2621791,2621792,client_e547b89c05043229,content_590b534cd83ded61,2026-02-01,0,no_clear_signal,0.0,0.0,NaN
2621792,2621793,client_e547b89c05043229,content_b82ea67f732ea382,2026-02-01,0,no_clear_signal,0.0,0.0,NaN



Why these picks may be weak:
Low-scoring rows have limited or no supporting GSC exposure/click signals. They should therefore be treated as lower-confidence review candidates rather than guaranteed refresh opportunities.

Leakage checks:
Scoring date range: 2026-02-01 to 2026-02-28
February-only scoring data: True
Scoring fields: ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position']

Leakage conclusion: No March/future-window fields were used in the baseline score.
Product-flag check: no product flags were used in the scoring rule.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.